In [1]:
!git clone https://github.com/huggingface/diffusers

Cloning into 'diffusers'...


In [2]:
cd diffusers

C:\Users\60124\diffusers


In [4]:
pip install -e .

Obtaining file:///C:/Users/60124/diffusers
  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     -------------------------------------- 302.0/302.0 kB 6.2 MB/s eta 0:00:00
     ------------------------------------- 277.2/277.2 kB 16.7 MB/s eta 0:00:00
     -------------------------------------- 166.4/166.4 kB 9.8 MB/s eta 0:00:00
  Building editable for diffusers (pyproject.toml): started
  Building editable for diffusers (pyproject.toml): finished 

In [8]:
cd examples/custom_diffusion

[WinError 3] The system cannot find the path specified: 'examples/custom_diffusion'
C:\Users\60124\diffusers\examples\custom_diffusion


In [9]:
pip install -r requirements.txt

     -------------------------------------- 261.4/261.4 kB 5.5 MB/s eta 0:00:00
     ---------------------------------------- 7.9/7.9 MB 2.4 MB/s eta 0:00:00
     ---------------------------------------- 53.1/53.1 kB 2.9 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 1.4 MB/s eta 0:00:00
     -------------------------------------- 295.0/295.0 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.18.0
    Uninstalling huggingface-hub-0.18.0:
      Successfully uninstalled huggingface-hub-0.18.0
Note: you may need to restart the kernel to use updated packages.


In [11]:
from accelerate.utils import write_basic_config

write_basic_config()

WindowsPath('C:/Users/60124/.cache/huggingface/accelerate/default_config.yaml')

In [30]:
# Define paths and variables
MODEL_NAME = "CompVis/stable-diffusion-v1-4"
OUTPUT_DIR = "C:\\Users\\60124\\Documents\\UM\\Thesis\\test"
INSTANCE_DIR = "C:\\Users\\60124\\Documents\\UM\\Thesis\\test"  # Change this to the path of your dataset directory
CSV_FILE = "dataset.csv"  # Name of your CSV file
CLASS_PROMPT = "metamaterial structure"  # Modify this according to your dataset
INSTANCE_PROMPT = "aluminium alloy"  # Modify this according to your dataset
MODIFIER_TOKEN = "structure"  # Modify as needed

In [29]:
!accelerate launch train_custom_diffusion.py \
    --pretrained_model_name_or_path=$MODEL_NAME \
    --instance_data_dir=$INSTANCE_DIR \
    --output_dir=$OUTPUT_DIR \
    --class_data_dir=$INSTANCE_DIR \
    --with_prior_preservation --real_prior --prior_loss_weight=1.0 \
    --class_prompt=$CLASS_PROMPT \
    --num_class_images=200 \
    --instance_prompt=$INSTANCE_PROMPT \
    --resolution=512 \
    --train_batch_size=2 \
    --learning_rate=1e-5 \
    --lr_warmup_steps=0 \
    --max_train_steps=250 \
    --scale_lr --hflip \
    --modifier_token $MODIFIER_PROMPT \
    --push_to_hub

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_cpu_threads_per_process` was set to `14` to improve out-of-box performance when training on CPUs
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
C:\Users\60124\anaconda3\python.exe: can't open file 'C:\Users\60124\train_custom_diffusion.py': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "C:\Users\60124\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\60124\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\60124\anaconda3\Scripts\accelerate.exe\__main__.py", line 7, in <module>
  File "C:\Users\60124\anaconda3\lib\site-packages\accelerate\commands\accelerate_cli.py", line 47, in main
    args.func(args)
  File "C:\Users\60124\anaconda3\lib\site-packages\accelerate\commands\launch.py", line 

In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModelForImageClassification, AutoFeatureExtractor

# Define your dataset directory and CSV file containing image paths, stress, and mass values
dataset_dir = "C:\\Users\\60124\\Documents\\UM\\Thesis\\test"
csv_file = "dataset.csv"

# Define image transformations
image_transforms = transforms.Compose([
    transforms.Resize((512, 512)),  # Adjust image size as needed
    transforms.ToTensor(),
])

# Load the CSV file containing image paths, stress, and mass values
data = pd.read_csv(os.path.join(dataset_dir, csv_file))

# Define a custom dataset class
class AtomicStructureDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = os.path.join(dataset_dir, self.data.iloc[idx, 0])
        image = Image.open(image_path)
        stress = float(self.data.iloc[idx, 1])
        mass = float(self.data.iloc[idx, 2])

        if self.transform:
            image = self.transform(image)

        return image, stress, mass

# Create an instance of the custom dataset
dataset = AtomicStructureDataset(data, transform=image_transforms)

# Create a DataLoader for batch processing
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Initialize the model and feature extractor
model = AutoModelForImageClassification.from_pretrained("CompVis/stable-diffusion-v1-4")
feature_extractor = AutoFeatureExtractor.from_pretrained("CompVis/stable-diffusion-v1-4")

# Define your training loop here
for batch in dataloader:
    images, stress_values, mass_values = batch

    # Forward pass through the model and compute losses
    # Modify this part based on your specific model architecture and loss functions

    # Example: Using stress and mass values as input
    inputs = {
        "pixel_values": images,
        "labels": torch.cat((stress_values.unsqueeze(1), mass_values.unsqueeze(1)), dim=1),
    }

    outputs = model(**inputs)

    # Backpropagation and optimization

# Your training loop will depend on the specific requirements of your model and task

# Save the trained model
output_dir = "C:\\Users\\60124\\Documents\\UM\\Thesis\\test"
model.save_pretrained(output_dir)

OSError: CompVis/stable-diffusion-v1-4 does not appear to have a file named config.json. Checkout 'https://huggingface.co/CompVis/stable-diffusion-v1-4/main' for available files.